In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_excel('tabella_36soggetti_fenotipi.xlsx') 
df_clean = df.dropna(subset=['Velocità_Hurried_cm/s']).copy()

df_train = df_clean[df_clean['Fenotipo'].isin(['Sani Puri (Resilienti)', 'Vuln. Globali'])].copy()

y = (df_train['Fenotipo'] == 'Vuln. Globali').astype(int)

X = df_train[['Età', 'MoCA', 'UPDRS_III', 'Velocità_Hurried_cm/s']]

# STANDARDIZZAZIONE ---
# Trasformiamo tutte le variabili nella stessa scala matematica
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# REGRESSIONE LOGISTICA ---
modello_reg = LogisticRegression(random_state=42)
modello_reg.fit(X_scaled, y)

# Prendiamo i coefficienti matematici estratti dal modello
coefficienti_puri = modello_reg.coef_[0]

pesi_assoluti = np.abs(coefficienti_puri)

pesi_percentuali = (pesi_assoluti / pesi_assoluti.sum()) * 100

nomi_variabili = X.columns
df_pesi = pd.DataFrame({
    'Variabile': nomi_variabili,
    'Coefficiente_Puro': coefficienti_puri.round(3),
    'Peso_IVP_(%)': pesi_percentuali.round(1)
})

# Ordiniamo dal peso più alto al più basso
df_pesi = df_pesi.sort_values(by='Peso_IVP_(%)', ascending=False)

print("\n=== CALCOLO DEI PESI PER L'INDICE IVP ===")
print(df_pesi.to_string(index=False))


=== CALCOLO DEI PESI PER L'INDICE IVP ===
            Variabile  Coefficiente_Puro  Peso_IVP_(%)
                 MoCA             -1.481          41.4
            UPDRS_III              1.136          31.8
Velocità_Hurried_cm/s             -0.772          21.6
                  Età             -0.184           5.2
